# Differential equations and rate laws

```{admonition} Learning outcomes
After working through this chapter, you should be able to:

1. explain how a differential equation describes change in a dynamic system
2. derive and implement the Forward Euler method
3. model simple and coupled chemical rate laws
4. investigate how the time step affects numerical error and stability
5. use `solve_ivp` to solve initial-value problems
6. validate a simulation using analytical solutions, mass balance and chemical plausibility
```

## Motivation

Experiments are central to chemistry, but simulations have become an important supplement. A simulation can help us investigate how a chemical system evolves when we change a parameter, test a model against experimental data, or study systems that are difficult to follow directly.

Often, we do not know a ready-made expression for the evolution, but we do know the **change**. A classic example is a rate law. For a first-order reaction

$$\mathrm{A\rightarrow products}$$

we have

$$\frac{d[A]}{dt}=-k[A].$$

The equation tells us how the concentration changes right now. Our task is to use this information to find the complete evolution, $[A](t)$.

## What is a differential equation?

A differential equation is an equation containing an unknown function and one or more derivatives of that function. You may encounter many forms:

$$y'=y$$

$$y'=t-y$$

$$u'(t)=u(t)$$

or, more generally,

$$y'(t)=\frac{dy}{dt}=f(t,y).$$

What they have in common is that the left-hand side describes the **change**, while the right-hand side tells us what the change depends on.

When we solve an ordinary algebraic equation, we search for a number. When we solve a differential equation, we search for a **function** or, numerically, a sequence of function values.

### Why do we need an initial value?

Consider the very simple differential equation

$$y'=1.$$

Integrating gives

$$y=t+C.$$

There are therefore infinitely many solutions – one for each value of the constant $C$. If we also know that

$$y(0)=2,$$

then $C=2$, and we obtain one particular solution. Such information is called an **initial condition**.

In chemistry, an initial condition may for example be the starting concentration $[A](0)$.

## From the derivative to Euler's method

From numerical differentiation, we know the forward difference:

$$\frac{dy}{dt}\approx\frac{y(t+\Delta t)-y(t)}{\Delta t}.$$

We now use it in a slightly different way. We know $y(t)$ and the expression for the derivative $dy/dt$, and we want to find the **next value**, $y(t+\Delta t)$.

First, multiply by $\Delta t$:

$$\frac{dy}{dt}\Delta t\approx y(t+\Delta t)-y(t).$$

Then move $y(t)$ to the other side:

$$y(t+\Delta t)\approx y(t)+\frac{dy}{dt}\Delta t.$$

If we write the differential equation as $dy/dt=f(t,y)$ and use indices, we obtain

$$y_{n+1}=y_n+f(t_n,y_n)\Delta t.$$

This is **Forward Euler**. Notice the connection to the previous numerical ideas: we have turned a continuous differential equation into a **difference equation** that the computer can repeat step by step.


## First-order reaction with Euler

We use

$$\frac{d[A]}{dt}=-k[A].$$

Before writing a general Euler function, we spell out the algorithm directly. This makes the connection between the rate law and the code as clear as possible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

k = 0.03
A0 = 1.0
dt = 1.0
t_end = 150

time = np.arange(0, t_end + dt, dt)
A = np.zeros(len(time))
A[0] = A0

for n in range(len(time) - 1):
    rate_of_change = -k*A[n]
    A[n + 1] = A[n] + rate_of_change*dt

plt.plot(time, A)
plt.xlabel("Time (s)")
plt.ylabel("[A] (mol/L)")
plt.show()


The loop follows Euler's formula directly: calculate the rate of change from the current concentration, multiply by the time step, and use that change to obtain the next concentration.

Once this update rule is clear, we can write a general Euler function. The function receives the differential equation itself as the function `f`.


In [ ]:
def first_order(t, A):
    return -k*A

def euler(f, y0, t_start, t_end, dt):
    time = np.arange(t_start, t_end + dt, dt)
    y = np.zeros(len(time))
    y[0] = y0

    for n in range(len(time) - 1):
        y[n + 1] = y[n] + f(time[n], y[n])*dt

    return time, y

time, A_euler = euler(first_order, A0, 0, 150, 1.0)
A_analytical = A0*np.exp(-k*time)

plt.plot(time, A_euler, label="Euler")
plt.plot(time, A_analytical, "--", label="Analytical")
plt.xlabel("Time (s)")
plt.ylabel("[A] (mol/L)")
plt.legend()
plt.show()

print("Largest absolute error:", np.max(np.abs(A_euler - A_analytical)))


## How large should the time step be?

Euler assumes that the slope we have **now** is a good approximation throughout the next time step. If the time step is large and the system changes rapidly, that assumption becomes poor.

We should therefore not investigate only one value of $\Delta t$. An important numerical check is:

> Does the solution change substantially if we reduce the time step?

Forward Euler is a **first-order method**, meaning that its global error decreases approximately in proportion to $\Delta t$ for a sufficiently smooth problem.


In [ ]:
for dt_test in [10.0, 5.0, 1.0, 0.2]:
    t_test, A_test = euler(first_order, A0, 0, 150, dt_test)
    A_exact = A0*np.exp(-k*t_test[-1])
    error = abs(A_test[-1] - A_exact)
    print(f"dt = {dt_test:4.1f} s   final error = {error:.3e}")


### Try it yourself

Change the rate constant and time step and investigate when Euler gives a good approximation.

<iframe src="../../basthon/?from=examples/differential_equation_euler_reaction.py" width="100%" height="640" frameborder="0" title="Try it yourself: Euler and reaction kinetics" loading="lazy" allowfullscreen></iframe>

## Coupled rate laws

Many chemical systems contain several concentrations that evolve at the same time. For the irreversible reaction

$$\mathrm{A+B\rightarrow C}$$

with rate

$$r=k[A][B],$$

we obtain

$$\frac{d[A]}{dt}=-k[A][B],$$

$$\frac{d[B]}{dt}=-k[A][B],$$

$$\frac{d[C]}{dt}=k[A][B].$$

The three equations are **coupled** because the same reaction rate depends on both $[A]$ and $[B]$.

In [ ]:
k = 0.05
A0 = 1.00
B0 = 0.60
C0 = 0.00
dt = 0.05
t_end = 80

t = np.arange(0, t_end + dt, dt)
A = np.zeros(len(t))
B = np.zeros(len(t))
C_prod = np.zeros(len(t))
A[0], B[0], C_prod[0] = A0, B0, C0

for n in range(len(t) - 1):
    rate = k*A[n]*B[n]
    A[n + 1] = A[n] - rate*dt
    B[n + 1] = B[n] - rate*dt
    C_prod[n + 1] = C_prod[n] + rate*dt

plt.plot(t, A, label="[A]")
plt.plot(t, B, label="[B]")
plt.plot(t, C_prod, label="[C]")
plt.xlabel("Time")
plt.ylabel("Concentration (mol/L)")
plt.legend()
plt.show()

### A chemical check: mass balance

A numerical curve can look smooth and still be wrong. Stoichiometry gives us independent checks. For $\mathrm{A+B\rightarrow C}$, every mole of C formed consumes one mole of A and one mole of B. Therefore, the model predicts that

$$[A]+[C]=[A]_0+[C]_0$$

and

$$[B]+[C]=[B]_0+[C]_0$$

should remain constant. These conservation relationships are powerful diagnostics for both the equations and the code.

In [ ]:
balance_A = A + C_prod
balance_B = B + C_prod

print("Largest deviation in the A balance:", np.max(np.abs(balance_A - A[0])))
print("Largest deviation in the B balance:", np.max(np.abs(balance_B - B[0])))

## Reversible reaction and dynamic equilibrium

For

$$\mathrm{A\rightleftharpoons B}$$

we can write

$$\frac{d[A]}{dt}=-k_f[A]+k_b[B],$$

$$\frac{d[B]}{dt}=k_f[A]-k_b[B].$$

When the system reaches dynamic equilibrium, the net change becomes zero even though the forward and backward reactions are still represented in the model.

In [ ]:
k_f = 0.40
k_b = 0.10
dt = 0.02
t = np.arange(0, 30 + dt, dt)

A = np.zeros(len(t))
B = np.zeros(len(t))
A[0] = 1.0

for n in range(len(t) - 1):
    net_rate = k_f*A[n] - k_b*B[n]
    A[n + 1] = A[n] - net_rate*dt
    B[n + 1] = B[n] + net_rate*dt

plt.plot(t, A, label="[A]")
plt.plot(t, B, label="[B]")
plt.xlabel("Time")
plt.ylabel("Concentration")
plt.legend()
plt.show()

print("A + B at the end:", A[-1] + B[-1])
print("B/A at the end:", B[-1]/A[-1])

At equilibrium, the net rate is zero,

$$k_f[A]=k_b[B],$$

so the model predicts

$$\frac{[B]}{[A]}=\frac{k_f}{k_b}.$$

The numerical result should approach this ratio. This gives another chemically meaningful check of the simulation.

## Higher-order methods: RK4

Forward Euler is a first-order method. **Order** describes how the numerical error decreases when the step size is reduced; it does not simply count how many times the function is evaluated.

The classical Runge–Kutta method RK4 uses four intermediate slopes in each step and has a global error that decreases approximately as $\Delta t^4$ when the assumptions are satisfied. It is therefore much more accurate than Euler for many smooth problems.

In [ ]:
def rk4_step(f, t, y, dt):
    k1 = f(t, y)
    k2 = f(t + dt/2, y + dt*k1/2)
    k3 = f(t + dt/2, y + dt*k2/2)
    k4 = f(t + dt, y + dt*k3)

    return y + dt*(k1 + 2*k2 + 2*k3 + k4)/6

def dy_dt(t, y):
    return -0.15*y

y = 1.0
t0 = 0.0
dt = 1.0

for n in range(10):
    y = rk4_step(dy_dt, t0 + n*dt, y, dt)

print("RK4 after 10 time units:", y)
print("Analytical:", np.exp(-0.15*10))

## `solve_ivp`: ready-made ODE solvers

In practical work, we normally use tested ODE solvers. `scipy.integrate.solve_ivp` solves initial-value problems and can adapt the time step automatically to keep the error below selected tolerances.

The default method is `RK45`, an adaptive Runge–Kutta method that uses an embedded pair of fifth and fourth order. The fifth-order formula is used for the step itself, while the difference between the two orders is used to estimate the error and control the step length.

In [ ]:
from scipy.integrate import solve_ivp

def first_order_rhs(t, y):
    return [-0.15*y[0]]

t_eval = np.linspace(0, 20, 201)

solution = solve_ivp(
    first_order_rhs,
    (0, 20),
    [1.0],
    t_eval=t_eval,
    rtol=1e-8,
    atol=1e-10
)

plt.plot(solution.t, solution.y[0], label="solve_ivp")
plt.plot(t_eval, np.exp(-0.15*t_eval), "--", label="Analytical")
plt.xlabel("Time")
plt.ylabel("[A]")
plt.legend()
plt.show()

print("Success:", solution.success)
print("Message:", solution.message)
print("Function evaluations:", solution.nfev)

### Coupled equations with `solve_ivp`

For a system of coupled equations, the function passed to `solve_ivp` returns one derivative for each state variable. The order of these derivatives must match the order of the concentrations in the state vector.

In [ ]:
def coupled_rhs(t, y):
    A, B, C = y
    k = 0.05
    rate = k*A*B
    return [-rate, -rate, rate]

solution = solve_ivp(
    coupled_rhs,
    (0, 80),
    [1.00, 0.60, 0.00],
    t_eval=np.linspace(0, 80, 401),
    rtol=1e-8,
    atol=1e-10
)

A, B, C_prod = solution.y

plt.plot(solution.t, A, label="[A]")
plt.plot(solution.t, B, label="[B]")
plt.plot(solution.t, C_prod, label="[C]")
plt.xlabel("Time")
plt.ylabel("Concentration (mol/L)")
plt.legend()
plt.show()

print("Largest deviation in A + C:", np.max(np.abs(A + C_prod - 1.00)))
print("Largest deviation in B + C:", np.max(np.abs(B + C_prod - 0.60)))

## Stiff systems

Chemical kinetic models can contain processes with very different time scales. A very fast reaction may force an explicit solver to take tiny time steps even after the fast transient is essentially over. Such a problem may be **stiff**.

Consider consecutive first-order reactions

$$\mathrm{A\xrightarrow{k_1}B\xrightarrow{k_2}C}$$

with $k_1=100$ and $k_2=0.02$. The first process is thousands of times faster than the second. `solve_ivp` provides methods designed for stiff problems, including `BDF` and `Radau`.

In [ ]:
def stiff_rhs(t, y):
    A, B, C = y
    k1 = 100.0
    k2 = 0.02
    return [-k1*A, k1*A - k2*B, k2*B]

rk45 = solve_ivp(stiff_rhs, (0, 200), [1, 0, 0], method="RK45",
                 rtol=1e-7, atol=1e-9)
bdf = solve_ivp(stiff_rhs, (0, 200), [1, 0, 0], method="BDF",
                rtol=1e-7, atol=1e-9)

print("RK45 function evaluations:", rk45.nfev)
print("BDF function evaluations:", bdf.nfev)
print("RK45 success:", rk45.success)
print("BDF success:", bdf.success)

The number of function evaluations is not a universal measure of speed, but a large difference can reveal that an explicit method is struggling with a stiff problem. The appropriate method depends on the model and the required accuracy.

## How should we validate a numerical model?

A successful solver call does not guarantee that the chemistry is correct. Numerical modelling should combine mathematical and chemical checks.

```{admonition} Validation checklist
:class: important
1. Have I formulated the correct rate laws and stoichiometric signs?
2. Are the units consistent?
3. Does the solution change substantially if I tighten the tolerance or reduce the time step?
4. Are amount of substance or other quantities conserved when the model says they should be?
5. Do limiting cases agree with chemical intuition?
```

## Short summary

- A differential equation describes the relationship between the state of a system and its change.
- $f'(t)$ and $df/dt$ are two notations for the derivative.
- Euler creates a discrete difference equation from a continuous rate of change.
- Coupled rate laws must be updated consistently with the stoichiometry.
- RK4 is a fourth-order method; “order” describes error scaling, not merely the number of function evaluations.
- `solve_ivp` is the natural tool for practical ODE problems.
- Stiff systems may require methods such as BDF or Radau.
- Numerical results should be checked using chemical conservation laws and convergence.

## Exercises

```{admonition} Exercise 1 – Euler and an analytical solution
:class: tip
Solve $d[A]/dt=-0.20[A]$ with $[A](0)=1.50$ mol/L using Euler. Compare with the analytical solution after 5, 10 and 20 time units.
```

```{admonition} Exercise 2 – time step
:class: tip
Repeat Exercise 1 with $\Delta t=2.0$, 1.0, 0.5, 0.1 and 0.01. Plot the error at $t=20$ as a function of $\Delta t$.
```

```{admonition} Exercise 3 – coupled reaction
:class: tip
Simulate $\mathrm{A+B\rightarrow C}$ with $k=0.05$, $[A]_0=1.00$ M and $[B]_0=0.60$ M. Check that $[A]+[C]$ and $[B]+[C]$ remain constant.
```

```{admonition} Exercise 4 – reversible reaction
:class: tip
For $\mathrm{A\rightleftharpoons B}$, use $k_f=0.40$ and $k_b=0.10$. Start with only A. Simulate until the system is close to equilibrium, and compare the numerical ratio $[B]/[A]$ with $k_f/k_b$.
```

```{admonition} Exercise 5 – Euler versus RK4
:class: tip
Use the same relatively large time step to solve a first-order reaction with Euler and RK4. Compare both with the analytical solution.
```

```{admonition} Exercise 6 – `solve_ivp`
:class: tip
Solve the first-order reaction with `solve_ivp`. Inspect the fields `success`, `message`, and the number of function evaluations `nfev`. What do they tell you?
```

```{admonition} Exercise 7 – consecutive reactions
:class: tip
Model $\mathrm{A\rightarrow B\rightarrow C}$ with two different rate constants. Find the time at which $[B]$ is largest. Explain chemically why the intermediate first increases and then decreases.
```

```{admonition} Exercise 8 – stiff system
:class: tip
Use the model with $k_1=100$ and $k_2=0.02$. Solve it with `RK45` and `BDF` using the same tolerances. Compare `nfev` and comment on the difference.
```